# Ariadne — lote LLM em GPU gratuita

Este notebook é **genérico**: lê pedidos de chat (`tasks.jsonl`), responde-lhes com um modelo aberto
no vLLM e escreve `results.jsonl`. Não sabe o que é curadoria, formatação ou domínios; isso é do
Ariadne. O caminho base é **só por ficheiros**: os pedidos entram (dataset do Kaggle ou pacote carregado
no Colab) e os resultados saem (output do Kaggle ou download no Colab). Se o manifesto trouxer `gateway`
(o túnel do Ariadne), o notebook também envia os resultados e o progresso pelas 4 rotas `/gpu/v1/…` dessa
corrida (com um token só dela) e, no pacote «magro» do Colab, puxa daí os pedidos. Se o túnel falhar, segue
só com ficheiros.

| | Kaggle (automático) | Colab (manual) |
|---|---|---|
| Arranque | o Ariadne faz `kaggle kernels push` | abres este notebook, escolhes GPU T4 e *Run all* |
| Entrada | dataset privado `ariadne-batch-<n>` | carregas o `.zip` que o painel do Ariadne gera (só o manifesto, se há túnel) |
| Saída | `results.jsonl` no output do kernel | descarrega `ariadne_results.zip` e carrega-o no painel |

**Modelo (T4 de 16 GB, sem bf16): `solidrust/Mistral-7B-Instruct-v0.3-AWQ`.**
- 4 bits (AWQ): ~4 GB de pesos; sobram ~9 GB para a cache KV. O Mistral 7B usa GQA (8 cabeças KV),
  ~128 KB por token, o que dá umas 60 mil tokens de cache: várias páginas de ~10 mil tokens em paralelo.
- A T4 (Turing) não tem bf16 nem FlashAttention 2: corre em fp16 com o *backend* xformers. O Mistral
  corre em fp16; o Gemma 3 do curador local não corre aqui (o vLLM recusa o `gemma3` em fp16 por instabilidade numérica).
- Licença Apache-2.0, sem acesso restrito. **Nenhum modelo chinês** (Qwen, DeepSeek, bge/BAAI, GLM, Yi, InternLM,
  MiniCPM, Baichuan): é regra do projeto e o notebook recusa-os (célula seguinte). É o modelo por omissão (escolhido pela fidelidade medida contra o
  Llama 3.1 8B, ver ARCHITECTURE.md); muda-se no painel do Ariadne (destino > modelo).

> Nota: o vLLM e o modelo descarregam-se da internet (o kernel tem a internet ligada só por isto).
> A versão do vLLM está fixa na célula seguinte; se o Kaggle mudar a imagem, é aí que se ajusta.

In [ ]:
import json
import os
import sys
import time
import zipfile
from pathlib import Path

T0 = time.time()
ON_KAGGLE = Path("/kaggle").exists()
ON_COLAB = ("google.colab" in sys.modules or Path("/content").exists()) and not ON_KAGGLE
WORK = Path("/kaggle/working") if ON_KAGGLE else Path("/content") if ON_COLAB else Path.cwd()
WORK.mkdir(parents=True, exist_ok=True)

VLLM_VERSION = "0.10.2"  # fixa. O Kaggle tem Python 3.13: o 0.8.5.post1 não tem roda para ele («No matching distribution»); o 0.10.2 é o 1.º com roda abi3 que ainda traz o motor V0 (a T4 não corre o V1)
TRANSFORMERS_VERSION = "4.56.2"  # o Kaggle traz transformers 5, em que o tokenizador de vários modelos perde `all_special_tokens_extended` e o vLLM 0.10 falha
DEFAULT_MODEL = "solidrust/Mistral-7B-Instruct-v0.3-AWQ"
# Regra do projeto: nenhum modelo chinês. Repete `ariadne/models_policy.py` (há um teste que compara as listas).
BLOCKED_PREFIXES = ("qwen/", "deepseek-ai/", "baai/", "thudm/", "01-ai/", "internlm/", "openbmb/", "baichuan-inc/")
BLOCKED_FAMILIES = ("qwen", "deepseek", "bge-", "glm-", "yi-", "internlm", "minicpm", "baichuan")
SETUP_MARGIN_SECONDS = 120  # parar este tempo antes do prazo para dar tempo a escrever e fechar
CHUNK = 96  # pedidos por lote enviado ao vLLM (cada um pode ter respostas longas)

In [ ]:
def locate_input() -> Path:
    """A pasta com tasks.jsonl + manifest.json: o dataset do Kaggle, o diretório atual, ou o
    .zip que se carrega no Colab."""
    roots = [Path("/kaggle/input"), Path("."), Path("/content"), WORK]
    for root in roots:
        if not root.exists():
            continue
        for manifest in sorted(root.rglob("manifest.json")):
            try:
                if "run_id" in json.loads(manifest.read_text(encoding="utf-8")):
                    return manifest.parent
            except ValueError:
                continue
    if ON_COLAB:
        from google.colab import files  # type: ignore[import-not-found]

        print("Carrega o pacote .zip gerado pelo painel do Ariadne (tasks.jsonl + manifest.json).")
        uploaded = files.upload()
        for name in uploaded:
            if name.endswith(".zip"):
                target = Path("/content/ariadne_input")
                with zipfile.ZipFile(name) as archive:
                    archive.extractall(target)
                return target
            if name == "manifest.json":
                return Path("/content")
    raise SystemExit("Não encontrei manifest.json (dataset do Kaggle ou pacote do Colab).")


INPUT = locate_input()
MANIFEST = json.loads((INPUT / "manifest.json").read_text(encoding="utf-8"))
MODEL = MANIFEST.get("model") or DEFAULT_MODEL
_name = MODEL.strip().lower()
if _name.startswith(BLOCKED_PREFIXES) or _name.rsplit("/", 1)[-1].startswith(BLOCKED_FAMILIES):
    raise SystemExit(f"Modelo recusado: {MODEL!r} é de uma família excluída do Ariadne (modelos chineses).")
MAX_MODEL_LEN = int(MANIFEST.get("max_model_len") or 24576)
DEADLINE = float(MANIFEST.get("deadline_seconds") or 6 * 3600)
RUN_ID = MANIFEST.get("run_id")
GATEWAY = MANIFEST.get("gateway")  # o túnel do Ariadne (opcional): ver a célula seguinte

In [ ]:
import time as _time
import urllib.error
import urllib.request


def gateway(method: str, path: str, data: bytes | None = None, content_type: str = "application/json"):
    """Fala com o gateway do Ariadne (só /gpu/v1/… desta corrida, com o token desta corrida). Sem
    túnel, ou se ele falhar, devolve None: o caminho por ficheiros continua a valer."""
    global GATEWAY
    if not GATEWAY:
        return None
    request = urllib.request.Request(
        f"{GATEWAY['url']}/gpu/v1/runs/{RUN_ID}/{path}", data=data, method=method
    )
    request.add_header("Authorization", "Bearer " + GATEWAY["token"])
    request.add_header("Content-Type", content_type)
    for key, value in (GATEWAY.get("headers") or {}).items():
        request.add_header(key, value)
    for attempt in range(4):
        try:
            with urllib.request.urlopen(request, timeout=60) as response:
                return response.read()
        except urllib.error.HTTPError as exc:
            if exc.code in (400, 401, 403, 404, 409, 410, 413):  # recusado: não adianta insistir
                print("Túnel recusou:", exc.code, "(sigo só com ficheiros)")
                GATEWAY = None
                return None
        except (urllib.error.URLError, TimeoutError, OSError):
            pass
        _time.sleep(2 ** attempt)
    print("Túnel indisponível (sigo só com ficheiros).")
    GATEWAY = None
    return None


def load_tasks() -> list[dict]:
    local = INPUT / "tasks.jsonl"
    if local.exists():
        text = local.read_text(encoding="utf-8")
    else:  # pacote «magro» do Colab: os pedidos vêm pelo túnel
        pages, offset = [], 0
        while True:
            chunk = gateway("GET", f"tasks?offset={offset}&limit=5000")
            if chunk is None:
                raise SystemExit("Sem tasks.jsonl e sem túnel: descarrega o pacote completo no painel.")
            lines = chunk.decode("utf-8").splitlines()
            pages.extend(lines)
            offset += len(lines)
            if len(lines) < 5000:
                break
        text = "\n".join(pages)
    return [json.loads(line) for line in text.splitlines() if line.strip()]


TASKS = load_tasks()
print(f"corrida {RUN_ID}: {len(TASKS)} pedidos, modelo {MODEL}, prazo {DEADLINE / 3600:.1f} h, túnel: {'sim' if GATEWAY else 'não'}")

In [ ]:
# Falha cedo e com uma mensagem clara: sem internet (DNS) ou sem GPU o resto só gasta tempo de quota.
import socket, subprocess
print("Python", sys.version.split()[0])
try:
    socket.gethostbyname("pypi.org")
except OSError as exc:
    raise SystemExit(f"Sem internet no kernel ({exc}). No Kaggle a internet e a GPU exigem a conta verificada por telefone.")
gpus = subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout.strip()
print(gpus or "nvidia-smi sem GPU")
if not gpus:
    raise SystemExit("Sem GPU no kernel. No Kaggle a GPU exige a conta verificada por telefone.")


In [ ]:
# Instala o vLLM (a imagem do Kaggle/Colab traz outro PyTorch; o vLLM traz o seu).
%pip install -q "vllm==$VLLM_VERSION" "transformers==$TRANSFORMERS_VERSION"

In [ ]:
os.environ["VLLM_USE_V1"] = "0"  # o motor V1 pede GPU Ampere; a T4 usa o V0
os.environ["VLLM_ATTENTION_BACKEND"] = "XFORMERS"

import torch
from vllm import LLM, SamplingParams

GPUS = torch.cuda.device_count()
assert GPUS >= 1, "Sem GPU: ativa um acelerador (T4) nas definições do notebook."
TP = int(MANIFEST.get("tensor_parallel_size") or 0) or GPUS
quant = "awq" if "AWQ" in MODEL.upper() else None
print("GPU:", torch.cuda.get_device_name(0), "x", GPUS, "| tensor parallel:", TP, "| quantização:", quant)

def load(tp: int) -> LLM:
    return LLM(
        model=MODEL,
        quantization=quant,
        dtype="half",  # a T4 não tem bf16
        max_model_len=MAX_MODEL_LEN,
        gpu_memory_utilization=0.90,
        tensor_parallel_size=tp,
        trust_remote_code=False,
    )

try:
    llm = load(TP)
except Exception as exc:  # NCCL / memória: tenta com uma só GPU
    print("Falhou com", TP, "GPU:", exc)
    if TP == 1:
        raise
    llm = load(1)
tokenizer = llm.get_tokenizer()

In [ ]:
RESULTS = WORK / "results.jsonl"
META = WORK / "results.meta.json"

def _accepts_system() -> bool:
    try:
        tokenizer.apply_chat_template([{"role": "system", "content": "s"}, {"role": "user", "content": "u"}], add_generation_prompt=True, tokenize=False)
        return True
    except Exception:  # p. ex. o modelo de conversa do Mistral v0.3 AWQ: «roles must alternate»
        return False


NO_SYSTEM = not _accepts_system()
print("modelo carregado:", MODEL, "| aceita papel system:", not NO_SYSTEM)


def fit(messages: list[dict]) -> list[dict]:
    """Se o modelo de conversa não aceita `system`, junta-o ao início da 1.ª mensagem do utilizador."""
    if not NO_SYSTEM or not messages or messages[0].get("role") != "system":
        return messages
    system, rest = messages[0]["content"], list(messages[1:])
    if rest and rest[0].get("role") == "user":
        rest[0] = {**rest[0], "content": system + "\n\n" + rest[0]["content"]}
        return rest
    return [{"role": "user", "content": system}, *rest]


def prompt_tokens(messages: list[dict]) -> int:
    return len(tokenizer.apply_chat_template(fit(messages), add_generation_prompt=True, tokenize=True))

done: set[str] = set()
if RESULTS.exists():  # retoma se o kernel foi reiniciado a meio
    for line in RESULTS.read_text(encoding="utf-8").splitlines():
        try:
            done.add(json.loads(line)["id"])
        except (ValueError, KeyError):
            pass

pending = [t for t in TASKS if t["id"] not in done]
stopped_by_deadline = False
written = 0
errors = 0

def write(handle, row: dict) -> None:
    line = json.dumps(row, ensure_ascii=False) + "\n"
    handle.write(line)
    SENT_BUFFER.append(line)


SENT_BUFFER: list[str] = []


def push(done_count: int) -> None:
    """Envia ao Ariadne o que ficou escrito desde o último envio, e diz como vai."""
    if not GATEWAY:
        SENT_BUFFER.clear()
        return
    if SENT_BUFFER:
        gateway("POST", "results", "".join(SENT_BUFFER).encode("utf-8"), "application/x-ndjson")
        SENT_BUFFER.clear()
    gateway("POST", "heartbeat", json.dumps({"done": done_count, "total": len(TASKS), "gpu_seconds": round(time.time() - T0, 1)}).encode())

with RESULTS.open("a", encoding="utf-8") as out:
    for start in range(0, len(pending), CHUNK):
        if time.time() - T0 > DEADLINE - SETUP_MARGIN_SECONDS - 600:  # sem tempo para outro lote
            stopped_by_deadline = True
            print("Prazo a chegar: paro aqui; o que falta volta ao Ariadne.")
            break
        batch = pending[start : start + CHUNK]
        chosen, runnable, params = [], [], []
        for task in batch:
            room = MAX_MODEL_LEN - prompt_tokens(task["messages"]) - 16
            wanted = int(task.get("max_tokens") or 4096)
            if room < 256:
                write(out, {"id": task["id"], "error": "pedido demasiado longo para o contexto"})
                errors += 1
                continue
            chosen.append(task)
            runnable.append(fit(task["messages"]))
            params.append(SamplingParams(temperature=float(task.get("temperature") or 0.0), max_tokens=min(wanted, room)))
        if runnable:
            outputs = llm.chat(runnable, params, use_tqdm=False)
            for task, output in zip(chosen, outputs):
                choice = output.outputs[0]
                write(out, {"id": task["id"], "content": choice.text, "finish_reason": choice.finish_reason,
                            "usage": {"prompt": len(output.prompt_token_ids), "completion": len(choice.token_ids)}})
                written += 1
        out.flush()
        push(len(done) + start + len(batch))
        print(f"{start + len(batch)}/{len(pending)} pedidos | {(time.time() - T0) / 60:.1f} min")

In [ ]:
elapsed = time.time() - T0
META.write_text(json.dumps({
    "run_id": RUN_ID,
    "model": MODEL,
    "gpu_seconds": round(elapsed, 1),
    "requests_total": len(TASKS),
    "requests_answered": written + len(done),
    "requests_failed": errors,
    "stopped_by_deadline": stopped_by_deadline,
}, indent=2), encoding="utf-8")
print(META.read_text())
push(written + len(done))
gateway("POST", "finish", json.dumps({"gpu_seconds": round(elapsed, 1)}).encode())

if ON_COLAB:
    bundle = WORK / "ariadne_results.zip"
    with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as archive:
        archive.write(RESULTS, "results.jsonl")
        archive.write(META, "results.meta.json")
    from google.colab import files  # type: ignore[import-not-found]

    files.download(str(bundle))  # carrega este ficheiro no painel do Ariadne (Importar resultados)